In [ ]:
!pip install bioservices
!pip install biopython

In [ ]:
import time
import requests
import pandas as pd
from Bio.KEGG import REST
from bioservices import KEGG
from bs4 import BeautifulSoup

In [ ]:
kegg_statistics = "https://www.genome.jp/kegg/docs/module_statistics.html"

In [ ]:
def fetch_kegg_statistics():
  resp = requests.get(kegg_statistics)
  resp.raise_for_status()
  # parse page
  soup = BeautifulSoup(resp.text, 'html.parser')
  tables = soup.find_all('table')
  # table with prokaryotes
  for tbl in tables:
      headers = [th.get_text(strip=True).lower() for th in tbl.find_all('th')]
      if 'prokaryotes' in headers:
          df = pd.read_html(str(tbl))[0]
          # ensure columns are clean
          df.rename(columns=lambda c: c.strip(), inplace=True)
          return df

  raise RuntimeError("Prokaryotes table not found on KEGG stats page.")

In [ ]:
def fetch_all_modules():
  #Fetch module info from kegg using REST
  handle = REST.kegg_list("module")
  data = handle.read()
  handle.close()
  modules = [line.split('\t') for line in data.strip().split('\n')]
  return pd.DataFrame(modules, columns=["Module ID", "Module Name"])

In [ ]:
def fetch_module_definition(module_id, kegg_client):
    #Fetch expressions for each module
    info = kegg_client.get(module_id)
    parsed = kegg_client.parse(info)
    return parsed.get('DEFINITION', '')

In [ ]:
def filter_prokaryote_modules(modules_df, stats_df):
    #Merge modules with stats and filter those present in prokaryotes.
    merged = modules_df.merge(stats_df, left_on='Module ID', right_on='Module', how='left')
    # filter by Prokaryotes count
    mask = merged['Prokaryotes'].fillna(0) >= 1
    return merged.loc[mask, :].reset_index(drop=True)

In [ ]:
#Fetch module and stats
modules_df = fetch_all_modules()
stats_df = fetch_kegg_statistics()
#Gather only prokaryotic modules
prok_modules = filter_prokaryote_modules(modules_df, stats_df)
#Retrieve expressions
kegg = KEGG()
exp = []
for mid in prok_modules['Module ID']:
    try:
        exp.append(fetch_module_definition(mid, kegg))
    except Exception as e:
        exp.append(None)
    #slowly fetch time
    time.sleep(0.4)
prok_modules['Definition'] = exp

output_path = '/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/KEGG_Graph_Generation/KEGG_Prokaryote_Modules_with_Definitions.xlsx'
prok_modules.to_excel(output_path, index=False)
print(f"Saved {len(prok_modules)} modules to {output_path}")